In [1]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
import random
import numpy as np

In [2]:
import sys
import os
sys.path.append(os.path.abspath("/home1/smaruj/pytorch_akita/"))

# from model import SeqNN
from model_v2_compatible import SeqNN

In [3]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [4]:
model = SeqNN()
model.load_state_dict(torch.load("/home1/smaruj/pytorch_akita/model_0_v2_finetuned_correctly.pt", map_location=device))
model.eval()

/tmp/SLURM_1508869/ipykernel_3646490/2744654124.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("/home1/smaruj/pytorch_akita/model_0_v2_

SeqNN(
  (stochastic_reverse_complement): StochasticReverseComplement()
  (stochastic_shift): StochasticShift()
  (conv_block_1): ConvBlock(
    (conv): Conv1d(4, 128, kernel_size=(15,), stride=(1,), padding=(7,), bias=False)
    (batch_norm): BatchNorm1d(128, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_tower): ConvTower(
    (conv_tower): Sequential(
      (0): ReLU()
      (1): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (4): ReLU()
      (5): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (6): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (7): MaxPool1d(kernel_size=2, stride=2, paddi

In [ ]:
# for all the frames

# class SimpleSequenceDataset(Dataset):
#     def __init__(self, directory_path, start_index=1, end_index=300):
#         self.directory_path = directory_path
#         self.start_index = start_index
#         self.end_index = end_index
#         self.indices = list(range(start_index, end_index + 1))  # inclusive range

#     def __len__(self):
#         return len(self.indices)

#     def __getitem__(self, idx):
#         index = self.indices[idx]
#         file_path = f"{self.directory_path}/seq_{index}.pt"
#         sequence = torch.load(file_path).squeeze(0)  # removes batch dimension
#         return sequence

In [5]:
# for a subset of frames

import re

class ListSequenceDataset(Dataset):
    def __init__(self, directory_path, file_list=None):
        self.directory_path = directory_path
        
        # If file_list is None, scan directory for matching files
        if file_list is None:
            file_list = [
                f for f in os.listdir(directory_path) 
                if f.startswith("seq_") and f.endswith(".pt")
            ]
        
        # Extract the numeric part from each filename
        self.files = sorted(
            file_list,
            key=lambda x: int(re.search(r"seq_(\d+)\.pt", x).group(1))
        )

        print(self.files)
        
    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_path = os.path.join(self.directory_path, self.files[idx])
        sequence = torch.load(file_path).squeeze(0)
        return sequence

In [6]:
BATCH_SIZE = 1

In [7]:
# dataset = SimpleSequenceDataset("/scratch1/smaruj/genomic_map_transformation/movie", start_index=1, end_index=206)

dataset = ListSequenceDataset("/scratch1/smaruj/genomic_map_transformation/movie")

loader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

['seq_0.pt', 'seq_1.pt', 'seq_2.pt', 'seq_3.pt', 'seq_6.pt', 'seq_52.pt', 'seq_59.pt', 'seq_69.pt', 'seq_71.pt', 'seq_82.pt', 'seq_84.pt', 'seq_87.pt', 'seq_90.pt', 'seq_92.pt', 'seq_94.pt', 'seq_97.pt', 'seq_101.pt', 'seq_102.pt', 'seq_103.pt', 'seq_113.pt', 'seq_117.pt', 'seq_119.pt', 'seq_130.pt', 'seq_131.pt', 'seq_135.pt', 'seq_140.pt', 'seq_194.pt', 'seq_209.pt', 'seq_219.pt', 'seq_255.pt']


In [8]:
def set_diag(matrix, value, k):
    """Set diagonal `k` of a matrix to `value`."""
    rows, cols = matrix.shape
    for i in range(rows):
        if 0 <= i + k < cols:
            matrix[i, i + k] = value

def from_upper_triu_batch(batch_vectors, matrix_len=512, num_diags=2):
    """Convert a batch of upper-triangular vectors into symmetric matrices with np.nan on diagonals."""
    if isinstance(batch_vectors, torch.Tensor):
        batch_vectors = batch_vectors.detach().cpu().numpy()

    batch_size = len(batch_vectors)
    matrices = np.zeros((batch_size, matrix_len, matrix_len), dtype=np.float32)

    triu_indices = np.triu_indices(matrix_len, num_diags)

    for i in range(batch_size):
        matrices[i][triu_indices] = batch_vectors[i][0,:]
        # Mirror to lower triangle
        matrices[i] = matrices[i] + matrices[i].T

        # Set diagonals to np.nan
        for k in range(-num_diags + 1, num_diags):
            set_diag(matrices[i], np.nan, k)

    return matrices  # shape: [B, 512, 512]

In [9]:
CTCF_PWM = "/home1/smaruj/IterativeMutagenesis/MA0139.1.meme"

In [10]:
def read_meme_pwm_as_numpy(filename):
    pwm_list = []  # List to store PWM rows
    
    with open(filename, 'r') as file:
        in_matrix_section = False
        
        for line in file:
            line = line.strip()
            
            # Check if we are reading the PWM matrix
            if line.startswith("letter-probability matrix"):
                in_matrix_section = True  # Start reading matrix data
                continue  # Skip this header line
            
            # If we are in the matrix section, process the rows
            if in_matrix_section and line:
                pwm_row = [float(value) for value in line.split()]  # Parse values
                pwm_list.append(pwm_row)  # Append to the PWM list
            
            # If we encounter a new MOTIF or the end of file, stop matrix reading
            if line.startswith("MOTIF") and in_matrix_section:
                break
    
    # Convert the list to a numpy array
    pwm_array = np.array(pwm_list)
    
    return pwm_array

In [11]:
pwm_CTCF = read_meme_pwm_as_numpy(CTCF_PWM)
pwm_CTCF_tensor = torch.from_numpy(pwm_CTCF.T).float()
motifs_dict = {"CTCF": pwm_CTCF_tensor}

In [12]:
from tangermeme.tools import fimo

In [13]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

In [14]:
VMIN, VMAX = -0.6, 0.6
MAX_CTCF_HITS = 3
NBINS = 512

In [15]:
def plot_frame(contact_map, ctcf_hits_df, seq_idx, save_path=None):
    # Bin starts
    ctcf_hits_df["bin"] = ctcf_hits_df["start"] // 2048
    ctcf_hits_df = ctcf_hits_df[(ctcf_hits_df["bin"] >= 64) & (ctcf_hits_df["bin"] < 576)]
    ctcf_hits_df["cropped_bin"] = ctcf_hits_df["bin"] - 64

    # Prepare tracks
    ctcf_plus = np.zeros(NBINS)
    ctcf_minus = np.zeros(NBINS)

    for _, row in ctcf_hits_df.iterrows():
        b = int(row["cropped_bin"])
        if row["strand"] == "+":
            ctcf_plus[b] += 1
        else:
            ctcf_minus[b] += 1

    # Create figure
    fig = plt.figure(figsize=(6, 7), dpi=100)
    gs = gridspec.GridSpec(2, 1, height_ratios=[4, 1])

    # Contact map
    ax0 = plt.subplot(gs[0])
    im = ax0.matshow(contact_map, cmap='RdBu_r', vmin=VMIN, vmax=VMAX)
    # ax0.set_title(f"Predicted contact map (seq {seq_idx})")
    ax0.axis("off")

    # CTCF track
    ax1 = plt.subplot(gs[1], sharex=ax0)
    x = np.arange(NBINS)
    ax1.bar(x, ctcf_plus, width=1.0, color='black', label='+ strand', alpha=0.7)
    ax1.bar(x, -ctcf_minus, width=1.0, color='red', label='− strand', alpha=0.7)
    ax1.set_xlim(0, NBINS)
    ax1.set_ylim(-MAX_CTCF_HITS, MAX_CTCF_HITS)
    ax1.axhline(0, color='gray', linewidth=0.5)
    ax1.set_ylabel("CTCF")
    ax1.set_xlabel("Bin (2048 bp)")
    # ax1.legend(loc='upper right', fontsize='small')

    # Fix layout
    plt.subplots_adjust(hspace=0.05)

    # Optional: Save figure
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', pad_inches=0.1)
        plt.close()
    else:
        plt.show()


In [16]:
# preds_all = []
global_idx = 0

for batch in loader:
    batch_gpu = batch.to(device)
    preds = model(batch_gpu).cpu()
    maps = from_upper_triu_batch(preds)
    
    hits = fimo.fimo(
            motifs=motifs_dict,
            sequences=batch,
            threshold=1e-4,
            reverse_complement=True
        )[0]
    
    for seq_idx in range(BATCH_SIZE):
        
        # plt.figure(figsize=(5, 5))
        # plt.matshow(maps[seq_idx].astype(np.float16), cmap='RdBu_r', vmin=-1.0, vmax=1.0)
        # plt.colorbar()
        # plt.show()
        
        seq_hits = hits[hits["sequence_name"] == seq_idx]
        seq_hits["bin"] = seq_hits["start"] // 2048
        seq_hits = seq_hits[(seq_hits["bin"] >= 64) & (seq_hits["bin"] < 576)]
        seq_hits["cropped_bin"] = seq_hits["bin"] - 64
        
        ctcf_plus = np.zeros(512)
        ctcf_minus = np.zeros(512)

        for _, row in seq_hits.iterrows():
            b = row["cropped_bin"]
            if row["strand"] == "+":
                ctcf_plus[b] += 1
            else:
                ctcf_minus[b] += 1
        
        save_path = f"/scratch1/smaruj/genomic_map_transformation/movie_frames/frame_{global_idx:03d}.png"
        plot_frame(maps[seq_idx].astype(np.float16), seq_hits, seq_idx, save_path=save_path)
        
        global_idx += 1

/tmp/SLURM_1508869/ipykernel_3646490/3767058522.py:29: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sequence = torch.load(file_path).squeeze(0)
/tmp/SLURM_1508869/ipykernel